In here we are planing to try some models to get some basi observations

In [74]:
import pandas as pd
import numpy as np
import re
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
import contractions

# Load Data

In [75]:
df_1 = pd.read_csv('../data/raw/goemotions_1.csv') 
df_2 = pd.read_csv('../data/raw/goemotions_2.csv')
df_3 = pd.read_csv('../data/raw/goemotions_3.csv')

df = pd.concat([df_1, df_2, df_3], ignore_index=True, axis=0)
df.head()

,text,id,author,subreddit,link_id,parent_id,created_utc,rater_id,example_very_unclear,admiration,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,That game hurt.,eew5j0j,Brdd9,nrl,t3_ajis4z,t1_eew18eq,1.548381e+09,1,False,0,...,0,0,0,0,0,0,0,1,0,0
1,>sexuality shouldn’t be a grouping category I...,eemcysk,TheGreen888,unpopularopinion,t3_ai4q37,t3_ai4q37,1.548084e+09,37,True,0,...,0,0,0,0,0,0,0,0,0,0
2,"You do right, if you don't care then fuck 'em!",ed2mah1,Labalool,confessions,t3_abru74,t1_ed2m7g7,1.546428e+09,37,False,0,...,0,0,0,0,0,0,0,0,0,1
3,Man I love reddit.,eeibobj,MrsRobertshaw,facepalm,t3_ahulml,t3_ahulml,1.547965e+09,18,False,0,...,1,0,0,0,0,0,0,0,0,0
4,"[NAME] was nowhere near them, he was by the Fa...",eda6yn6,American_Fascist713,starwarsspeculation,t3_ackt2f,t1_eda65q2,1.546669e+09,2,False,0,...,0,0,0,0,0,0,0,0,0,1


# Data Preprocessing

In [76]:
def basic_clean(text):
  text = text.lower()
  text = re.sub(r'[^a-z\s]', '', text)
  return text

df['clean_text'] = df['text'].apply(basic_clean)
df['clean_text'] = df['clean_text'].apply(lambda x: contractions.fix(x))
df.head()

# Remove auxiliary verbs
aux_verbs = set(['be', 'am', 'is', 'are', 'was', 'were', 'being', 'been',
                 'have', 'has', 'had', 'having',
                 'do', 'does', 'did', 'doing',
                 'will', 'would', 'shall', 'should', 'may', 'might', 'must', 'can', 'could'])
def remove_aux_verbs(text):
    tokens = text.split()
    filtered_tokens = [word for word in tokens if word not in aux_verbs]
    return ' '.join(filtered_tokens)
df['clean_text'] = df['clean_text'].apply(remove_aux_verbs)

In [77]:
stemmer = PorterStemmer()
def stem_text(text):
  return " ".join([stemmer.stem(word) for word in text.split()])

df['stemmed'] = df['clean_text'].apply(stem_text)

# Target Processing

In [78]:
emotions = ['admiration',
       'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
       'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
       'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy',
       'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
       'remorse', 'sadness', 'surprise']
# Let's turn the these labels into a 3 labels dataset: positive, negative, neutral
sentiment_dict = pd.read_json('../data/sentiment_dict.json', typ='series').to_dict()
# In this sentiment dictionary, we have the following mapping:
#{
#"positive": ["amusement", "excitement", "joy", "love", "desire", "optimism", "caring", "pride", "admiration", "gratitude", "relief", "approval"],
#"negative": ["fear", "nervousness", "remorse", "embarrassment", "disappointment", "sadness", "grief", "disgust", "anger", "annoyance", "disapproval"],
#"ambiguous": ["realization", "surprise", "curiosity", "confusion"]
#}
def label_emotion(row):
    active_emotions = [emotion for emotion in emotions if row.get(emotion, 0) == 1]

    if not active_emotions:
        return 'neutral'

    sentiment_counts = {
        'positive': 0,
        'negative': 0,
        'ambiguous': 0
    }

    for emotion in active_emotions:
        for sentiment, emotion_list in sentiment_dict.items():
            if emotion in emotion_list:
                sentiment_counts[sentiment] += 1

    # Decide based on majority
    max_sentiment = max(sentiment_counts, key=sentiment_counts.get)

    # Optional: handle ties
    # 
    if list(sentiment_counts.values()).count(sentiment_counts[max_sentiment]) > 1:
       return 'mixed'

    return max_sentiment

df['sentiment'] = df.apply(label_emotion, axis=1)
# let's remove records with 'neutral' sentiment to focus on clear positive and negative examples
df = df[df['sentiment'] != 'neutral']
df = df[df['sentiment'] != 'mixed']

sentiment_mapping = {
       'ambiguous':0,
       'positive':1,
       'negative':2
}
df['sentiment_label'] = df['sentiment'].map(sentiment_mapping)
df.head()

y = df['sentiment_label']


# Feature Vectorization

In [79]:
# Convert text to TF-IDF features
X = df['stemmed']
vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.8)
X_vect = vectorizer.fit_transform(X)

# Feature Selection

In [80]:
vrthr = VarianceThreshold(threshold=0.0001)
X_vect = vrthr.fit_transform(X_vect)

# Train Test split of the data

In [81]:
X_tr, X_te, y_tr, y_te = train_test_split(
        X_vect, y.values, test_size=0.2, random_state=42, stratify=y.values
)

# Model Selection

In [82]:
models = {
  'Logistic Regression': LogisticRegression(random_state=42,solver='newton-cg', max_iter=1000),
  'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

In [83]:
results = []
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_te)

    acc = accuracy_score(y_te, y_pred)
    recall = recall_score(y_te,y_pred, average='macro')
    precision = precision_score(y_te, y_pred, average='macro')
    f1 = f1_score(y_te, y_pred, average='macro')

    results.append({
        'Model': name,
        'Accuracy': acc,
        'recall': recall,
        'precision': precision,
        'f1 score': f1
    })

    print(f'Model: {name} - Accuracy: {acc}, recall: {recall}, precision: {precision}, f1 score: {f1}')

results_df = pd.DataFrame(results)
results_df

Training Logistic Regression...
Model: Logistic Regression - Accuracy: 0.721005475360876, recall: 0.630970700908352, precision: 0.6829864357270775, f1 score: 0.6449205426139647
Training Random Forest...
Model: Random Forest - Accuracy: 0.7522576975040888, recall: 0.6894985222141264, precision: 0.7091578600539842, f1 score: 0.697512443916715


,Model,Accuracy,recall,precision,f1 score
0,Logistic Regression,0.721005,0.630971,0.682986,0.644921
1,Random Forest,0.752258,0.689499,0.709158,0.697512
